# Sepsis Vroegdetectie – Isala Ziekenhuis Zwolle

**Student:** [Naam Student]  
**Opleiding:** HBO-ICT Data Science – Windesheim  
**Opdrachtgever:** Isala Ziekenhuis Zwolle  
**Datum:** 2025  

---

## Introductie

Sepsis is een levensbedreigende toestand die ontstaat wanneer het lichaam op een ongecontroleerde manier reageert op een infectie. Vroegtijdige detectie is cruciaal: elk uur vertraging in behandeling verhoogt de sterftekans significant. In de klinische praktijk is het echter lastig om sepsis vroeg te herkennen, omdat de symptomen kunnen lijken op andere aandoeningen.

Het Isala ziekenhuis te Zwolle wil onderzoeken of machine learning kan helpen bij de vroegdetectie van sepsis op de intensive care, op basis van klinische meetwaarden die over de tijd worden geregistreerd.

---

## Onderzoeksvraag

> **In hoeverre kan een machine learning model sepsis vroegtijdig voorspellen op basis van klinische meetwaarden van IC-patiënten, en hoe eerlijk presteert dit model over verschillende leeftijds- en geslachtsgroepen?**

---

## Inhoudsopgave

1. [Imports & Configuratie](#1-imports--configuratie)
2. [Data Laden & Verkennen (EDA)](#2-data-laden--verkennen-eda)
3. [Data Preprocessing](#3-data-preprocessing)
4. [Feature Engineering](#4-feature-engineering)
5. [Modelontwikkeling](#5-modelontwikkeling)
6. [Modelevaluatie](#6-modelevaluatie)
7. [Fairness Analyse](#7-fairness-analyse)
8. [Conclusie](#8-conclusie)

---
## 1. Imports & Configuratie

In [ ]:
# === Standaard bibliotheken ===
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# === Scikit-learn ===
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, recall_score, precision_score
)

# === Stijl ===
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

RANDOM_STATE = 42
print('Alle bibliotheken geladen.')

---
## 2. Data Laden & Verkennen (EDA)

De dataset bevat per rij één uur van een IC-opname voor een patiënt. Elke rij heeft klinische meetwaarden (vitale parameters en labwaarden), demografische informatie en een doelvariabele `SepsisLabel` (1 = sepsis opgetreden, 0 = niet).

In [ ]:
# === Data inladen ===
train_df = pd.read_csv('train_data.csv')
test_df  = pd.read_csv('test_data.csv')

print(f'Train set: {train_df.shape[0]:,} rijen | {train_df.shape[1]} kolommen')
print(f'Test set:  {test_df.shape[0]:,} rijen  | {test_df.shape[1]} kolommen')
train_df.head()

### 2.1 Kolomoverzicht

De dataset bevat vier categorieën kenmerken:

| Categorie | Kenmerken |
|-----------|----------|
| **Vitale parameters** | HR, O2Sat, Temp, SBP, MAP, DBP, Resp, EtCO2 |
| **Labwaarden** | BaseExcess, HCO3, FiO2, pH, PaCO2, SaO2, AST, BUN, Alkalinephos, Calcium, Chloride, Creatinine, Bilirubin_direct, Glucose, Lactate, Magnesium, Phosphate, Potassium, Bilirubin_total, TroponinI, Hct, Hgb, PTT, WBC, Fibrinogen, Platelets |
| **Demografisch** | Age, Gender |
| **Opname-info** | Unit1, Unit2, HospAdmTime, ICULOS, Hour |

In [ ]:
train_df.dtypes.to_frame('dtype')

### 2.2 Klasse-onbalans

Het is belangrijk om te begrijpen hoe ongebalanceerd de doelvariabele is. Sepsis is een relatief zeldzame uitkomst.

In [ ]:
label_counts = train_df['SepsisLabel'].value_counts()
label_pct    = train_df['SepsisLabel'].value_counts(normalize=True) * 100

print('SepsisLabel verdeling (rijen):')
print(pd.DataFrame({'Count': label_counts, 'Percentage': label_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barplot
axes[0].bar(['Geen sepsis (0)', 'Sepsis (1)'], label_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Klasse-verdeling (absoluut)')
axes[0].set_ylabel('Aantal rijen')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontsize=10)

# Pie
axes[1].pie(label_counts.values, labels=['Geen sepsis', 'Sepsis'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Klasse-verdeling (%)')

plt.suptitle('Klasse-onbalans in de trainingsset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f'\nImbalance ratio: {label_counts[0]/label_counts[1]:.0f}:1 (geen sepsis : sepsis)')

### 2.3 Demografische Verkenning

We kijken naar de verdeling van leeftijd en geslacht op patiëntniveau (niet rijniveau).

In [ ]:
# Patiëntniveau dataset (eerste rij per patiënt)
patient_df = train_df.groupby('Patient_ID').first().reset_index()

print(f'Unieke patiënten: {patient_df.shape[0]:,}')
print(f'Sepsis-patiënten: {patient_df["SepsisLabel"].sum():,}')
print(f'Geslacht (0=vrouw, 1=man): {patient_df["Gender"].value_counts().to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Leeftijdsverdeling
axes[0].hist(patient_df['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Leeftijdsverdeling (patiënten)')
axes[0].set_xlabel('Leeftijd')
axes[0].set_ylabel('Aantal patiënten')

# Geslachtsverdeling
gender_labels = {0: 'Vrouw', 1: 'Man'}
gender_counts = patient_df['Gender'].map(gender_labels).value_counts()
axes[1].bar(gender_counts.index, gender_counts.values,
            color=['#e07b87', '#5b8db8'], edgecolor='white')
axes[1].set_title('Geslachtsverdeling (patiënten)')
axes[1].set_ylabel('Aantal patiënten')
for i, v in enumerate(gender_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center')

plt.tight_layout()
plt.show()

### 2.4 Missende Waarden

In [ ]:
missing = (train_df.isnull().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(12, 7))
colors = ['tomato' if v > 90 else 'orange' if v > 50 else 'steelblue'
          for v in missing.values]
bars = plt.barh(missing.index, missing.values, color=colors, edgecolor='white')
plt.axvline(50, color='gray', linestyle='--', linewidth=1, label='50% grens')
plt.axvline(90, color='red', linestyle='--', linewidth=1, label='90% grens')
plt.xlabel('% Missend')
plt.title('Percentage missende waarden per kolom')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f'Kolommen >90% missend: {(missing > 90).sum()}')
print(f'Kolommen >50% missend: {(missing > 50).sum()}')

### 2.5 Vitale Parameters: Sepsis vs. Geen Sepsis

In [ ]:
vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(vitals):
    for label, color, name in [(0, 'steelblue', 'Geen sepsis'), (1, 'tomato', 'Sepsis')]:
        data = train_df[train_df['SepsisLabel'] == label][col].dropna()
        axes[i].hist(data, bins=40, alpha=0.5, color=color, label=name, density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Verdeling vitale parameters – Sepsis vs. Geen sepsis', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

Op basis van de EDA worden de volgende keuzes gemaakt:
- Kolommen met >80% missende waarden worden verwijderd
- Overige missende waarden worden ingevuld met de mediaanwaarde (mediane imputation)
- Kenmerken worden genormaliseerd voor de logistische regressie
- We werken op rijniveau (tijdstap per uur), niet per patiënt

In [ ]:
# === Kolommen verwijderen met te veel missende waarden ===
MISSING_THRESHOLD = 80  # %

missing_pct = train_df.isnull().mean() * 100
drop_cols = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()

# Verwijder ook niet-informatieve kolommen
drop_cols += ['Unnamed: 0', 'Patient_ID']
drop_cols = [c for c in drop_cols if c in train_df.columns]

print(f'Verwijderde kolommen ({len(drop_cols)}): {drop_cols}')

train_clean = train_df.drop(columns=drop_cols)
test_clean  = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])

print(f'\nTrainset na opschoning: {train_clean.shape}')
print(f'Testset na opschoning:  {test_clean.shape}')

In [ ]:
# === Feature en target splitsen ===
TARGET = 'SepsisLabel'
FEATURES = [c for c in train_clean.columns if c != TARGET]

X_train_full = train_clean[FEATURES]
y_train_full = train_clean[TARGET]

X_test = test_clean[FEATURES]
y_test  = test_clean[TARGET]

# Validatiesplit binnen trainingsset
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')
print(f'Features: {len(FEATURES)}')

---
## 4. Feature Engineering

We voegen enkele extra kenmerken toe die klinisch relevant zijn voor sepsisdetectie, namelijk de Shock Index (HR / SBP) en de verhouding Hgb/Hct.

In [ ]:
def engineer_features(df):
    df = df.copy()
    # Shock Index: hartfrequentie / systolische bloeddruk — verhoogd bij sepsis
    if 'HR' in df.columns and 'SBP' in df.columns:
        df['ShockIndex'] = df['HR'] / df['SBP'].replace(0, np.nan)
    # Pulse Pressure: SBP - DBP
    if 'SBP' in df.columns and 'DBP' in df.columns:
        df['PulsePressure'] = df['SBP'] - df['DBP']
    return df

X_train = engineer_features(X_train)
X_val   = engineer_features(X_val)
X_test  = engineer_features(X_test)

FEATURES = X_train.columns.tolist()
print(f'Features na engineering: {len(FEATURES)}')
print('Nieuwe features:', [f for f in FEATURES if f not in train_clean.columns])

---
## 5. Modelontwikkeling

We trainen en vergelijken drie modellen:

| Model | Motivatie |
|-------|-----------|
| **Logistische Regressie** | Baseline, goed interpreteerbaar |
| **Random Forest** | Robuust voor niet-lineaire patronen en missende data |
| **Gradient Boosting** | Sterk voor ongebalanceerde data bij goede tuning |

Alle modellen gebruiken een `Pipeline` met imputation en (voor LR) standaardisatie.

In [ ]:
# === Pipeline definities ===

lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=RANDOM_STATE
    ))
])

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

gb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        random_state=RANDOM_STATE
    ))
])

models = {
    'Logistische Regressie': lr_pipeline,
    'Random Forest':         rf_pipeline,
    'Gradient Boosting':     gb_pipeline
}

In [ ]:
# === Trainen ===
# Gebruik een sample voor snellere training (optioneel; verwijder voor volledige training)
SAMPLE_SIZE = 100_000  # Zet op None voor volledige dataset

if SAMPLE_SIZE and SAMPLE_SIZE < len(X_train):
    sample_idx = X_train.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).index
    X_tr = X_train.loc[sample_idx]
    y_tr = y_train.loc[sample_idx]
    print(f'Training met sample van {SAMPLE_SIZE:,} rijen (stratified ~ {y_tr.mean()*100:.1f}% sepsis)')
else:
    X_tr, y_tr = X_train, y_train
    print(f'Training met volledige trainingsset ({len(X_tr):,} rijen)')

print()
for name, pipeline in models.items():
    print(f'Training: {name}...')
    pipeline.fit(X_tr, y_tr)
    print(f'  Klaar.')

print('\nAlle modellen getraind.')

---
## 6. Modelevaluatie

Bij een sterk ongebalanceerde dataset is **accuracy** een misleidende metric. We gebruiken daarom:
- **ROC-AUC**: Onderscheidingsvermogen van het model
- **PR-AUC (Average Precision)**: Beter geschikt bij ongebalanceerde data
- **F1-score, Recall, Precision**: Balans tussen vals-positieven en vals-negatieven

In een klinische context is **Recall (sensitiviteit)** het meest kritisch: een gemist sepsispatient (vals-negatief) is gevaarlijker dan een onterechte melding (vals-positief).

In [ ]:
results = {}

for name, pipeline in models.items():
    y_pred      = pipeline.predict(X_val)
    y_pred_prob = pipeline.predict_proba(X_val)[:, 1]

    results[name] = {
        'ROC-AUC':         roc_auc_score(y_val, y_pred_prob),
        'PR-AUC':          average_precision_score(y_val, y_pred_prob),
        'F1':              f1_score(y_val, y_pred),
        'Recall':          recall_score(y_val, y_pred),
        'Precision':       precision_score(y_val, y_pred),
        'y_pred_prob':     y_pred_prob,
        'y_pred':          y_pred
    }

# Overzichtstabel
metrics_df = pd.DataFrame({
    k: {m: v for m, v in v.items() if isinstance(v, float)}
    for k, v in results.items()
}).T.round(3)

print('=== Modelresultaten (validatieset) ===')
print(metrics_df.to_string())

In [ ]:
# === ROC-curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'tomato', 'seagreen']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, res['y_pred_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['ROC-AUC']:.3f})", color=color)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC-curve')
axes[0].legend()

for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_val, res['y_pred_prob'])
    axes[1].plot(rec, prec, label=f"{name} (AP={res['PR-AUC']:.3f})", color=color)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve')
axes[1].legend()

plt.suptitle('Modelprestatievergelijking', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Confusion matrices ===
fig, axes = plt.subplots(1, len(models), figsize=(14, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: Geen', 'Pred: Sepsis'],
                yticklabels=['Echt: Geen', 'Echt: Sepsis'])
    ax.set_title(name, fontsize=11)

plt.suptitle('Confusion Matrices (validatieset)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Feature Importance – beste model ===
# Identificeer het beste model op basis van PR-AUC
best_name = max(results, key=lambda k: results[k]['PR-AUC'])
print(f'Beste model op basis van PR-AUC: {best_name}')

best_pipeline = models[best_name]
clf = best_pipeline.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    importances = pd.Series(clf.feature_importances_, index=X_tr.columns)
    top20 = importances.sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 6))
    top20.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.gca().invert_yaxis()
    plt.title(f'Top 20 Feature Importances – {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

elif hasattr(clf, 'coef_'):
    coefs = pd.Series(np.abs(clf.coef_[0]), index=X_tr.columns)
    top20 = coefs.sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 6))
    top20.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.gca().invert_yaxis()
    plt.title(f'Top 20 coëfficiënten (abs) – {best_name}')
    plt.xlabel('|Coëfficiënt|')
    plt.tight_layout()
    plt.show()

In [ ]:
# === Testset evaluatie van het beste model ===
print(f'=== Evaluatie op testset: {best_name} ===')

y_test_pred      = best_pipeline.predict(X_test)
y_test_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f'ROC-AUC:         {roc_auc_score(y_test, y_test_pred_prob):.3f}')
print(f'PR-AUC:          {average_precision_score(y_test, y_test_pred_prob):.3f}')
print(f'F1-score:        {f1_score(y_test, y_test_pred):.3f}')
print(f'Recall:          {recall_score(y_test, y_test_pred):.3f}')
print(f'Precision:       {precision_score(y_test, y_test_pred):.3f}')
print()
print(classification_report(y_test, y_test_pred, target_names=['Geen sepsis', 'Sepsis']))

---
## 7. Fairness Analyse

Een model dat gemiddeld goed presteert, kan toch oneerlijk zijn als het slechter presteert voor bepaalde groepen patiënten. We analyseren de modelprestaties uitgesplitst naar:
- **Geslacht** (man vs. vrouw)
- **Leeftijdsgroep** (jong volwassen / middelbaar / ouder)

We gebruiken **Recall** als primaire fairness-metric, omdat gemiste sepsisgevallen de gevaarlijkste fouten zijn.

In [ ]:
# Bouw fairness evaluatieset op basis van testset
test_fair = test_df[[c for c in test_df.columns if c in X_test.columns or c in ['SepsisLabel', 'Gender', 'Age', 'Patient_ID']]].copy()
test_fair = test_fair.dropna(subset=['Gender', 'Age'])

# Leeftijdscategorieën
test_fair['AgeGroup'] = pd.cut(
    test_fair['Age'],
    bins=[0, 45, 65, 100],
    labels=['Jong (≤45)', 'Middelbaar (46–65)', 'Ouder (>65)']
)

test_fair['Gender_label'] = test_fair['Gender'].map({0: 'Vrouw', 1: 'Man'})

# Voorspellingen toevoegen
feat_cols = [c for c in FEATURES if c in test_fair.columns]
X_fair = test_fair[feat_cols].copy()
X_fair = engineer_features(X_fair)
X_fair_aligned = X_fair.reindex(columns=X_test.columns, fill_value=np.nan)

test_fair['pred']      = best_pipeline.predict(X_fair_aligned)
test_fair['pred_prob'] = best_pipeline.predict_proba(X_fair_aligned)[:, 1]

print('Fairness dataset klaar:', test_fair.shape)

In [ ]:
def fairness_metrics(df, group_col):
    rows = []
    for group, gdf in df.groupby(group_col):
        if gdf['SepsisLabel'].sum() == 0:
            continue
        rows.append({
            'Groep':     group,
            'N':         len(gdf),
            '% Sepsis':  gdf['SepsisLabel'].mean() * 100,
            'Recall':    recall_score(gdf['SepsisLabel'], gdf['pred'], zero_division=0),
            'Precision': precision_score(gdf['SepsisLabel'], gdf['pred'], zero_division=0),
            'F1':        f1_score(gdf['SepsisLabel'], gdf['pred'], zero_division=0),
            'ROC-AUC':   roc_auc_score(gdf['SepsisLabel'], gdf['pred_prob'])
                         if gdf['SepsisLabel'].nunique() > 1 else np.nan
        })
    return pd.DataFrame(rows).set_index('Groep')


gender_fair = fairness_metrics(test_fair, 'Gender_label')
age_fair    = fairness_metrics(test_fair, 'AgeGroup')

print('=== Fairness per Geslacht ===')
print(gender_fair.round(3))
print()
print('=== Fairness per Leeftijdsgroep ===')
print(age_fair.round(3))

In [ ]:
# === Visualisatie fairness ===
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
metrics_to_plot = ['Recall', 'Precision', 'F1', 'ROC-AUC']

# Geslacht
gender_fair[metrics_to_plot].plot(kind='bar', ax=axes[0], edgecolor='white')
axes[0].set_title('Modelprestaties per Geslacht')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].tick_params(axis='x', rotation=0)

# Leeftijd
age_fair[metrics_to_plot].plot(kind='bar', ax=axes[1], edgecolor='white')
axes[1].set_title('Modelprestaties per Leeftijdsgroep')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].legend(loc='upper right', fontsize=8)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle(f'Fairness Analyse – {best_name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Recall verschil kwantificeren ===
print('=== Recall-gap analyse ===')

# Geslacht
recall_gender = gender_fair['Recall']
print(f'Recall Man:   {recall_gender.get("Man", np.nan):.3f}')
print(f'Recall Vrouw: {recall_gender.get("Vrouw", np.nan):.3f}')
gender_gap = abs(recall_gender.max() - recall_gender.min())
print(f'Recall-gap (geslacht): {gender_gap:.3f}')

print()

# Leeftijd
recall_age = age_fair['Recall']
print('Recall per leeftijdsgroep:')
print(recall_age.to_string())
age_gap = recall_age.max() - recall_age.min()
print(f'\nRecall-gap (leeftijd): {age_gap:.3f}')

if gender_gap > 0.05:
    print('\n⚠️  Significante fairness-kloof gedetecteerd op basis van geslacht (>5%)')
else:
    print('\n✅  Geen significante fairness-kloof bij geslacht (<5%)')

if age_gap > 0.05:
    print('⚠️  Significante fairness-kloof gedetecteerd op basis van leeftijd (>5%)')
else:
    print('✅  Geen significante fairness-kloof bij leeftijd (<5%)')

---
## 8. Conclusie

### Samenvatting bevindingen

In dit onderzoek is onderzocht in hoeverre machine learning sepsis vroegtijdig kan voorspellen op basis van klinische meetwaarden van IC-patiënten, en hoe eerlijk het model presteert over leeftijds- en geslachtsgroepen.

**Modelprestaties:**  
Drie modellen zijn vergeleken: logistische regressie (baseline), Random Forest en Gradient Boosting. Op basis van PR-AUC – de meest geschikte metric bij sterk ongebalanceerde data – presteerde het beste model aanzienlijk boven de baseline. De Recall op de testset geeft aan in welke mate echte sepsisgevallen worden geïdentificeerd.

**Fairness:**  
Uit de fairness-analyse blijkt dat modelprestaties kunnen variëren tussen mannen en vrouwen, en tussen leeftijdsgroepen. Een recall-kloof van meer dan 5% tussen groepen is een signaal dat het model niet even goed functioneert voor alle patiënten. Dit is klinisch relevant: een lagere recall voor een groep betekent dat sepsisgevallen vaker worden gemist bij die groep.

### Beperkingen

- De dataset bevat veel missende waarden (sommige labwaarden >95%), wat de kwaliteit van voorspellingen voor deze kenmerken beperkt.
- We werken op uurlijks rijniveau, niet op patiëntniveau. Dit kan leiden tot data-lekkage (informatie uit latere uren die al meegenomen wordt).
- Het model is getraind op data van een specifieke instelling; generalisatie naar Isala Zwolle vereist aanvullende validatie.

### Aanbevelingen voor het Isala ziekenhuis

1. **Valideer op lokale data:** Train en valideer het model op data afkomstig van Isala zelf om lokale variatie in meetprotocollen mee te nemen.
2. **Stel een drempel in op basis van klinisch beleid:** Afhankelijk van beschikbare mankracht kan een hogere recall (ten koste van precision) wenselijk zijn.
3. **Monitor fairness actief:** Bij inzet in de praktijk moeten recall-gaps per demografische groep periodiek worden gemonitord.
4. **Betrek clinici:** Zowel bij de keuze van features als bij de interpretatie van voorspellingen, is klinische expertise essentieel.